# 04 Model Training

Train tuned baseline models, compare them with a weighted hybrid and a stacked ensemble, and save report-ready tables and figures.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from evaluation import (
    blend_probabilities,
    build_precision_recall_curve_df,
    evaluate_predictions,
    evaluate_with_best_f1_threshold,
    find_threshold_for_target_precision,
)
from models import (
    tune_lightgbm_baseline,
    tune_logistic_regression_baseline,
    tune_mlp_baseline,
    tune_random_forest_baseline,
)

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"
tables_dir = PROJECT_ROOT / "outputs" / "tables"
figures_dir = PROJECT_ROOT / "outputs" / "figures"
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

train_linear = pd.read_parquet(processed_dir / "train_linear_ready.parquet")
train_tree = pd.read_parquet(processed_dir / "train_tree_ready.parquet")

print("Linear-ready shape:", train_linear.shape)
print("Tree-ready shape:", train_tree.shape)

Linear-ready shape: (307511, 612)
Tree-ready shape: (307511, 612)


## Tuned Baseline Models

In [3]:
# Active Logistic Regression grid search:
log_reg_param_grid = {
    "C": [0.1, 1.0, 5.0],
}

# Frozen one-config fallback:
# log_reg_param_grid = {
#     "C": [5.0],
# }

log_reg_model, log_reg_results, y_train_log_reg, log_reg_train_probs, y_val_log_reg, log_reg_probs, log_reg_tuning_results, best_log_reg_params = tune_logistic_regression_baseline(
    param_grid=log_reg_param_grid
)

print("Best Logistic Regression params:", best_log_reg_params)
print("Logistic Regression Results:")
for metric, value in log_reg_results.items():
    if metric != "model":
        print(f"{metric}: {value:.4f}")


Best Logistic Regression params: {'C': 1.0}
Logistic Regression Results:
accuracy: 0.7119
precision: 0.1760
recall: 0.6973
f1: 0.2810
roc_auc: 0.7754
average_precision: 0.2659


In [4]:
# Active Random Forest grid search:
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [12, 20, None],
    "min_samples_leaf": [5, 10],
    "min_samples_split": [10, 20],
}

# Frozen one-config fallback:
# rf_param_grid = {
#     "n_estimators": [400],
#     "max_depth": [None],
#     "min_samples_leaf": [10],
#     "min_samples_split": [10],
# }

rf_model, rf_results, y_train_rf, rf_train_probs, y_val_rf, rf_probs, rf_tuning_results, best_rf_params = tune_random_forest_baseline(
    param_grid=rf_param_grid
)

print("Best Random Forest params:", best_rf_params)
print("Random Forest Results:")
for metric, value in rf_results.items():
    if metric != "model":
        print(f"{metric}: {value:.4f}")


Best Random Forest params: {'max_depth': None, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 400}
Random Forest Results:
accuracy: 0.9112
precision: 0.3804
recall: 0.1585
f1: 0.2238
roc_auc: 0.7709
average_precision: 0.2552


In [ ]:
# Active LightGBM grid search:
lgbm_param_grid = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.05],
    "num_leaves": [31, 64, 96],
    "min_child_samples": [20, 50],
}

# Frozen one-config fallback:
# lgbm_param_grid = {
#     "learning_rate": [0.03],
#     "min_child_samples": [50],
#     "n_estimators": [500],
#     "num_leaves": [64],
# }

lgbm_model, lgbm_results, y_train_lgbm, lgbm_train_probs, y_val_lgbm, lgbm_probs, lgbm_tuning_results, best_lgbm_params = tune_lightgbm_baseline(
    param_grid=lgbm_param_grid
)

print("Best LightGBM params:", best_lgbm_params)
print("LightGBM Results:")
for metric, value in lgbm_results.items():
    if metric != "model":
        print(f"{metric}: {value:.4f}")


In [ ]:
# Active Neural Network grid search:
mlp_param_grid = {
    "hidden_dims": [(256, 128), (128, 64), (256, 64)],
    "dropout": [0.2, 0.3],
    "learning_rate": [0.001, 0.0005],
    "epochs": [15, 20],
}

# Frozen one-config fallback:
# mlp_param_grid = {
#     "hidden_dims": [(128, 64)],
#     "dropout": [0.3],
#     "learning_rate": [0.0005],
#     "epochs": [15],
# }

mlp_model, mlp_results, mlp_history, y_train_mlp, mlp_train_probs, y_val_mlp, mlp_probs, mlp_tuning_results, best_mlp_params = tune_mlp_baseline(
    param_grid=mlp_param_grid
)

print("Best Neural Network params:", best_mlp_params)
print("Neural Network Results:")
for metric, value in mlp_results.items():
    if metric != "model":
        print(f"{metric}: {value:.4f}")

mlp_history.tail()


## Weighted Hybrid

The weighted hybrid is the project's novel contribution. It blends tuned Logistic Regression and tuned LightGBM validation probabilities with a fixed weight sweep.

In [ ]:
hybrid_y_val = y_val_log_reg.reset_index(drop=True)
if not hybrid_y_val.equals(y_val_lgbm.reset_index(drop=True)):
    raise ValueError("Validation labels do not align between Logistic Regression and LightGBM.")

hybrid_rows = []
for lgbm_weight in [0.50, 0.60, 0.70, 0.80, 0.90]:
    hybrid_probs_candidate = blend_probabilities(lgbm_probs, log_reg_probs, primary_weight=lgbm_weight)
    default_metrics = evaluate_predictions(hybrid_y_val, hybrid_probs_candidate)
    tuned_metrics = evaluate_with_best_f1_threshold(hybrid_y_val, hybrid_probs_candidate)
    hybrid_rows.append(
        {
            "lgbm_weight": lgbm_weight,
            "log_reg_weight": 1 - lgbm_weight,
            "roc_auc": default_metrics["roc_auc"],
            "average_precision": default_metrics["average_precision"],
            "default_f1": default_metrics["f1"],
            "tuned_f1": tuned_metrics["f1"],
            "tuned_precision": tuned_metrics["precision"],
            "tuned_recall": tuned_metrics["recall"],
            "tuned_threshold": tuned_metrics["threshold"],
        }
    )

hybrid_weight_df = pd.DataFrame(hybrid_rows)
hybrid_weight_df = hybrid_weight_df.sort_values(["roc_auc", "tuned_f1"], ascending=False).reset_index(drop=True)
hybrid_weight_df

In [ ]:
best_hybrid_row = hybrid_weight_df.iloc[0]
best_lgbm_weight = float(best_hybrid_row["lgbm_weight"])
hybrid_probs = blend_probabilities(lgbm_probs, log_reg_probs, primary_weight=best_lgbm_weight)

hybrid_results = {
    "model": f"Hybrid (LGBM {best_lgbm_weight:.1f} + LR {1 - best_lgbm_weight:.1f})",
    **evaluate_predictions(hybrid_y_val, hybrid_probs),
}

hybrid_tuned_results = {
    "model": hybrid_results["model"],
    **evaluate_with_best_f1_threshold(hybrid_y_val, hybrid_probs),
}

pd.Series(hybrid_tuned_results)

## Stacked Ensemble

The stacked ensemble uses train-side base-model probabilities as meta-features and fits a simple Logistic Regression stacker.

In [ ]:
stack_y_train = y_train_log_reg.reset_index(drop=True)
stack_y_val = y_val_log_reg.reset_index(drop=True)

stack_train_features = pd.DataFrame({
    "log_reg_prob": log_reg_train_probs.reset_index(drop=True),
    "rf_prob": rf_train_probs.reset_index(drop=True),
    "lgbm_prob": lgbm_train_probs.reset_index(drop=True),
    "mlp_prob": mlp_train_probs.reset_index(drop=True),
})

stack_val_features = pd.DataFrame({
    "log_reg_prob": log_reg_probs.reset_index(drop=True),
    "rf_prob": rf_probs.reset_index(drop=True),
    "lgbm_prob": lgbm_probs.reset_index(drop=True),
    "mlp_prob": mlp_probs.reset_index(drop=True),
})

stack_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
stack_model.fit(stack_train_features, stack_y_train)
stack_probs = pd.Series(stack_model.predict_proba(stack_val_features)[:, 1])

stacked_results = {
    "model": "Stacked Ensemble",
    **evaluate_predictions(stack_y_val, stack_probs),
}

stacked_tuned_results = {
    "model": "Stacked Ensemble",
    **evaluate_with_best_f1_threshold(stack_y_val, stack_probs),
}

pd.Series(stacked_tuned_results)

## Model Comparison

In [ ]:
log_reg_tuned_results = {
    "model": "Logistic Regression",
    **evaluate_with_best_f1_threshold(y_val_log_reg, log_reg_probs),
}

rf_tuned_results = {
    "model": "Random Forest",
    **evaluate_with_best_f1_threshold(y_val_rf, rf_probs),
}

lgbm_tuned_results = {
    "model": "LightGBM",
    **evaluate_with_best_f1_threshold(y_val_lgbm, lgbm_probs),
}

mlp_tuned_results = {
    "model": "Neural Network",
    **evaluate_with_best_f1_threshold(y_val_mlp, mlp_probs),
}

tuned_results_df = pd.DataFrame([
    log_reg_tuned_results,
    rf_tuned_results,
    lgbm_tuned_results,
    mlp_tuned_results,
    hybrid_tuned_results,
    stacked_tuned_results,
])
tuned_results_df = tuned_results_df.sort_values(["average_precision", "roc_auc", "f1"], ascending=False).reset_index(drop=True)
tuned_results_df

In [ ]:
results_df = pd.DataFrame([
    log_reg_results,
    rf_results,
    lgbm_results,
    mlp_results,
    hybrid_results,
    stacked_results,
])
results_df = results_df.sort_values(["average_precision", "roc_auc", "f1"], ascending=False).reset_index(drop=True)
results_df

In [ ]:
results_df.to_csv(tables_dir / "model_metrics_default.csv", index=False)
tuned_results_df.to_csv(tables_dir / "model_metrics_tuned.csv", index=False)
hybrid_weight_df.to_csv(tables_dir / "hybrid_weight_sweep.csv", index=False)
log_reg_tuning_results.to_csv(tables_dir / "logistic_regression_tuning_results.csv", index=False)
rf_tuning_results.to_csv(tables_dir / "random_forest_tuning_results.csv", index=False)
lgbm_tuning_results.to_csv(tables_dir / "lightgbm_tuning_results.csv", index=False)
mlp_tuning_results.to_csv(tables_dir / "neural_net_tuning_results.csv", index=False)

print(tables_dir / "model_metrics_default.csv")
print(tables_dir / "model_metrics_tuned.csv")
print(tables_dir / "hybrid_weight_sweep.csv")
print(tables_dir / "logistic_regression_tuning_results.csv")
print(tables_dir / "random_forest_tuning_results.csv")
print(tables_dir / "lightgbm_tuning_results.csv")
print(tables_dir / "neural_net_tuning_results.csv")

In [ ]:
comparison_plot_df = results_df.set_index("model")[["roc_auc", "average_precision", "f1"]]

ax = comparison_plot_df.plot(kind="bar", figsize=(10, 6))
ax.set_title("Model Comparison on Validation Split")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
curve_dfs = {
    "Logistic Regression": build_precision_recall_curve_df(y_val_log_reg, log_reg_probs),
    "Random Forest": build_precision_recall_curve_df(y_val_rf, rf_probs),
    "LightGBM": build_precision_recall_curve_df(y_val_lgbm, lgbm_probs),
    "Neural Network": build_precision_recall_curve_df(y_val_mlp, mlp_probs),
    hybrid_results["model"]: build_precision_recall_curve_df(hybrid_y_val, hybrid_probs),
    stacked_results["model"]: build_precision_recall_curve_df(stack_y_val, stack_probs),
}

plt.figure(figsize=(8, 6))
for model_name, curve_df in curve_dfs.items():
    plt.plot(curve_df["recall"], curve_df["precision"], label=model_name)

baseline_precision = y_val_lgbm.mean()
plt.axhline(baseline_precision, linestyle="--", color="gray", label="Base rate")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "precision_recall_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
pd.DataFrame([
    find_threshold_for_target_precision(y_val_lgbm, lgbm_probs, target_precision=0.30),
    find_threshold_for_target_precision(hybrid_y_val, hybrid_probs, target_precision=0.30),
    find_threshold_for_target_precision(stack_y_val, stack_probs, target_precision=0.30),
], index=["LightGBM", hybrid_results["model"], stacked_results["model"]])

## Short Conclusions

- All main models now receive actual hyperparameter tuning, with RF and the neural net tuned on grids designed to be comparable in breadth to LightGBM.
- Treat the weighted hybrid as the project's novel contribution and compare it directly against tuned standalone LightGBM and tuned standalone Logistic Regression.
- Treat the stacked ensemble as an additional low-churn ensemble baseline that combines all four tuned model outputs.
- Use `model_metrics_tuned.csv` for the fairest overall comparison.
